In [ ]:
 # ═══════════════════════════════════════════════════════════════════════════════
# PHASE 4 — REGIME-CONDITIONAL SIMILARITY
# Inputs:  outputs/02_returns_matrix.parquet
#          outputs/06_tail_dependence_lower.parquet
#          outputs/subperiod_labels.csv
#          outputs/03_coin_metadata.csv
# Outputs: outputs/08_regime_labels.csv               ← date → bull/bear label
#          outputs/09_regime_conditional_corr.parquet ← ρ_bull, ρ_bear per pair
#          outputs/phase4_regime_stability.csv        ← regime stability gap per pair
#          outputs/phase4_markov_params.csv           ← model parameters
# ═══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

from scipy.stats import norm, spearmanr
from scipy.optimize import minimize
from statsmodels.stats.multitest import multipletests

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

Path("outputs").mkdir(exist_ok=True)
Path("outputs/figures").mkdir(exist_ok=True)


# ═══════════════════════════════════════════════════════════════════════════════
# LOAD INPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("Loading Phase 3 outputs...")

log_returns  = pd.read_parquet("outputs/02_returns_matrix.parquet")
lam_L_mat    = pd.read_parquet("outputs/06_tail_dependence_lower.parquet")
subperiod_df = pd.read_csv("outputs/subperiod_labels.csv", parse_dates=["date"])
coin_meta    = pd.read_csv("outputs/03_coin_metadata.csv")

coins = log_returns.columns.tolist()
N     = len(coins)

print(f"  Coins         : {N}")
print(f"  Return matrix : {log_returns.shape}")
print(f"  Date range    : {log_returns.index[0]}  →  {log_returns.index[-1]}")


# ═══════════════════════════════════════════════════════════════════════════════
# 4.1  MARKET REGIME IDENTIFICATION
# Markov Regime Switching Model on BTC log returns
# ═══════════════════════════════════════════════════════════════════════════════
# Why BTC as the regime signal?
# BTC is the market factor — its volatility state defines the macro environment
# for the entire crypto universe. When BTC enters a high-volatility bear regime,
# nearly all coins are affected. Using BTC returns to define regimes means our
# regime labels are market-wide, not coin-specific.
#
# We fit a 2-state Markov Switching model:
#   State 1 (Bull)  : low volatility, positive drift
#   State 2 (Bear)  : high volatility, negative or flat drift
#
# Then test a 3-state model (Bull / Sideways / Bear) and compare.
# The Hamilton Filter extracts daily regime probabilities.
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 4.1 │ Markov Regime Switching Model ───────────────────────────────")

btc_returns = log_returns["bitcoin"].dropna().values
btc_dates   = log_returns["bitcoin"].dropna().index
T           = len(btc_returns)

print(f"  BTC return series length : {T} days")


# ── Hamilton Filter ───────────────────────────────────────────────────────────
def hamilton_filter(returns: np.ndarray,
                    mu: np.ndarray,
                    sigma: np.ndarray,
                    P: np.ndarray) -> tuple:
    """
    Hamilton (1989) filter for Markov Regime Switching model.

    Parameters
    ----------
    returns : T-length return series
    mu      : (K,) mean returns per state
    sigma   : (K,) std devs per state
    P       : (K, K) transition probability matrix

    Returns
    -------
    filtered_probs  : (T, K) filtered state probabilities P(S_t | Y_1..t)
    log_likelihood  : scalar total log-likelihood
    """
    K          = len(mu)
    T          = len(returns)
    xi         = np.full((T, K), np.nan)   # filtered probs
    eta        = np.full((T, K), np.nan)   # conditional densities

    # Initialise with stationary distribution
    # π = (I - P' + 1)^(-1) * 1  (stationary dist of Markov chain)
    try:
        A      = np.eye(K) - P.T + np.ones((K, K))
        pi0    = np.linalg.solve(A, np.ones(K))
        pi0    = np.clip(pi0, 1e-6, 1)
        pi0   /= pi0.sum()
    except np.linalg.LinAlgError:
        pi0    = np.ones(K) / K

    log_lik = 0.0

    for t in range(T):
        # Conditional densities: p(y_t | S_t = k)
        for k in range(K):
            eta[t, k] = norm.pdf(returns[t], loc=mu[k], scale=max(sigma[k], 1e-8))

        if t == 0:
            xi_prev = pi0
        else:
            xi_prev = xi[t - 1]

        # Prediction: P(S_t = k | Y_1..t-1)
        xi_pred = P.T @ xi_prev

        # Update: P(S_t = k | Y_1..t)
        numer   = eta[t] * xi_pred
        denom   = numer.sum()
        if denom < 1e-300:
            xi[t] = xi_pred    # degenerate — keep prediction
        else:
            xi[t]    = numer / denom
            log_lik += np.log(denom)

    return xi, log_lik


# ── MRS parameter estimation via EM-like numerical optimisation ───────────────
def fit_mrs(returns: np.ndarray, K: int = 2,
            n_restarts: int = 8) -> dict:
    """
    Fit K-state Markov Regime Switching model by maximising log-likelihood.

    Parameters
    ----------
    returns    : return series
    K          : number of states (2 or 3)
    n_restarts : random restarts to avoid local optima

    Returns
    -------
    dict with mu, sigma, P, filtered_probs, log_likelihood, AIC, BIC
    """
    T   = len(returns)
    best_ll  = -np.inf
    best_res = None

    def pack(mu, log_sigma, logit_p):
        return np.concatenate([mu, log_sigma, logit_p.ravel()])

    def unpack(params):
        mu_      = params[:K]
        sigma_   = np.exp(params[K:2*K]) + 1e-6
        # Transition probabilities: softmax row-wise
        raw_p    = params[2*K:].reshape(K, K)
        P_       = np.zeros((K, K))
        for k in range(K):
            exp_row     = np.exp(raw_p[k] - raw_p[k].max())
            P_[k]       = exp_row / exp_row.sum()
        return mu_, sigma_, P_

    def neg_loglik(params):
        try:
            mu_, sigma_, P_ = unpack(params)
            _, ll = hamilton_filter(returns, mu_, sigma_, P_)
            return -ll if np.isfinite(ll) else 1e10
        except Exception:
            return 1e10

    np.random.seed(42)
    for restart in range(n_restarts):
        # Random initialisation — spread means across return distribution
        mu_init      = np.percentile(returns,
                                     np.linspace(20, 80, K)) + \
                       np.random.randn(K) * returns.std() * 0.3
        log_sig_init = np.log(np.abs(returns.std()) *
                              np.linspace(0.5, 2.0, K) + 1e-6) + \
                       np.random.randn(K) * 0.2
        logit_p_init = np.random.randn(K, K) * 0.5

        x0 = pack(mu_init, log_sig_init, logit_p_init)

        res = minimize(neg_loglik, x0,
                       method="Nelder-Mead",
                       options={"maxiter": 5000, "xatol": 1e-5,
                                "fatol": 1e-5, "adaptive": True})

        if -res.fun > best_ll:
            best_ll  = -res.fun
            best_res = res

    if best_res is None:
        return None

    mu_, sigma_, P_ = unpack(best_res.x)
    filtered_probs, ll = hamilton_filter(returns, mu_, sigma_, P_)

    # Sort states by volatility ascending → State 0 = bull, State K-1 = bear
    order  = np.argsort(sigma_)
    mu_    = mu_[order]
    sigma_ = sigma_[order]
    P_     = P_[np.ix_(order, order)]
    filtered_probs = filtered_probs[:, order]

    n_params = K * 2 + K * K   # mu, sigma, transition probs
    aic      = 2 * n_params - 2 * ll
    bic      = n_params * np.log(T) - 2 * ll

    return {
        "K"              : K,
        "mu"             : mu_,
        "sigma"          : sigma_,
        "P"              : P_,
        "filtered_probs" : filtered_probs,
        "log_likelihood" : ll,
        "aic"            : aic,
        "bic"            : bic,
    }


# ─── Fit 2-state model ────────────────────────────────────────────────────────
print("\n  Fitting 2-state MRS model (Bull / Bear)...")
mrs2 = fit_mrs(btc_returns, K=2, n_restarts=3)

print(f"    State 0 (Bull) : μ = {mrs2['mu'][0]:.5f}  σ = {mrs2['sigma'][0]:.5f}")
print(f"    State 1 (Bear) : μ = {mrs2['mu'][1]:.5f}  σ = {mrs2['sigma'][1]:.5f}")
print(f"    Transition P   :")
print(f"      Bull→Bull : {mrs2['P'][0,0]:.4f}   Bull→Bear : {mrs2['P'][0,1]:.4f}")
print(f"      Bear→Bull : {mrs2['P'][1,0]:.4f}   Bear→Bear : {mrs2['P'][1,1]:.4f}")
print(f"    Log-likelihood : {mrs2['log_likelihood']:.2f}")
print(f"    AIC            : {mrs2['aic']:.2f}")
print(f"    BIC            : {mrs2['bic']:.2f}")

# ─── Fit 3-state model ────────────────────────────────────────────────────────
print("\n  Fitting 3-state MRS model (Bull / Sideways / Bear)...")
mrs3 = fit_mrs(btc_returns, K=3, n_restarts=3)

print(f"    State 0 (Bull)     : μ = {mrs3['mu'][0]:.5f}  σ = {mrs3['sigma'][0]:.5f}")
print(f"    State 1 (Sideways) : μ = {mrs3['mu'][1]:.5f}  σ = {mrs3['sigma'][1]:.5f}")
print(f"    State 2 (Bear)     : μ = {mrs3['mu'][2]:.5f}  σ = {mrs3['sigma'][2]:.5f}")
print(f"    Log-likelihood     : {mrs3['log_likelihood']:.2f}")
print(f"    AIC                : {mrs3['aic']:.2f}")
print(f"    BIC                : {mrs3['bic']:.2f}")

# ─── Model comparison ─────────────────────────────────────────────────────────
print(f"\n  Model comparison:")
print(f"    2-state BIC : {mrs2['bic']:.2f}")
print(f"    3-state BIC : {mrs3['bic']:.2f}")

if mrs2['bic'] <= mrs3['bic']:
    print(f"    → 2-state model preferred (lower BIC)")
    best_mrs  = mrs2
    n_states  = 2
    state_names = ["Bull", "Bear"]
else:
    print(f"    → 3-state model preferred (lower BIC)")
    best_mrs  = mrs3
    n_states  = 3
    state_names = ["Bull", "Sideways", "Bear"]

print(f"\n  Using {n_states}-state model for regime labels")


# ═══════════════════════════════════════════════════════════════════════════════
# 4.2  EXTRACT DAILY REGIME LABELS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 4.2 │ Daily Regime Labels ─────────────────────────────────────────")

# Assign dominant regime: state with highest filtered probability
# We require probability > 0.65 for a clean assignment; below that → "uncertain"
PROB_THRESHOLD = 0.65

filtered_probs = best_mrs["filtered_probs"]   # (T, K)
dominant_state = np.argmax(filtered_probs, axis=1)
dominant_prob  = filtered_probs[np.arange(T), dominant_state]

regime_labels  = []
for s, p in zip(dominant_state, dominant_prob):
    if p >= PROB_THRESHOLD:
        regime_labels.append(state_names[s])
    else:
        regime_labels.append("Uncertain")

regime_df = pd.DataFrame({
    "date"          : btc_dates,
    "regime"        : regime_labels,
    "dominant_state": dominant_state,
    "dominant_prob" : dominant_prob,
})
for k, name in enumerate(state_names):
    regime_df[f"prob_{name.lower()}"] = filtered_probs[:, k]

# ─── Regime distribution ─────────────────────────────────────────────────────
regime_counts = regime_df["regime"].value_counts()
print(f"\n  Regime distribution:")
for regime, cnt in regime_counts.items():
    pct = cnt / T * 100
    print(f"    {regime:12s} : {cnt:3d} days  ({pct:.1f}%)")

# ─── Regime persistence (avg consecutive days in each state) ──────────────────
for regime in state_names:
    mask    = (regime_df["regime"] == regime).values
    runs    = []
    run_len = 0
    for val in mask:
        if val:
            run_len += 1
        elif run_len > 0:
            runs.append(run_len)
            run_len = 0
    if run_len > 0:
        runs.append(run_len)
    avg_run = np.mean(runs) if runs else 0
    print(f"    Avg consecutive {regime:10s} days : {avg_run:.1f}")

# ─── Plot regime probabilities ────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# BTC returns
axes[0].plot(btc_dates, btc_returns, color="steelblue", linewidth=0.5, alpha=0.8)
axes[0].set_ylabel("BTC Log Return")
axes[0].set_title("BTC Returns with Markov Regime Identification")
axes[0].axhline(0, color="gray", linewidth=0.5)

# Bull probability
axes[1].fill_between(btc_dates,
                     filtered_probs[:, 0],
                     alpha=0.7, color="green", label="P(Bull)")
axes[1].axhline(PROB_THRESHOLD, color="black", linestyle="--",
                linewidth=0.8, label=f"Threshold ({PROB_THRESHOLD})")
axes[1].set_ylabel("P(Bull)")
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)

# Bear probability
bear_idx = n_states - 1
axes[2].fill_between(btc_dates,
                     filtered_probs[:, bear_idx],
                     alpha=0.7, color="red", label="P(Bear)")
axes[2].axhline(PROB_THRESHOLD, color="black", linestyle="--",
                linewidth=0.8, label=f"Threshold ({PROB_THRESHOLD})")
axes[2].set_ylabel("P(Bear)")
axes[2].set_ylim(0, 1)
axes[2].legend(fontsize=8)

# Mark consolidated subperiod breaks
for brk in [pd.Timestamp("2024-11-01"),
            pd.Timestamp("2025-02-01"),
            pd.Timestamp("2025-10-01")]:
    for ax in axes:
        ax.axvline(brk, color="orange", linestyle="--",
                   alpha=0.6, linewidth=0.8)

plt.tight_layout()
plt.savefig("outputs/figures/phase4_regime_probabilities.png", dpi=150)
plt.close()
print("\n  ✅ Saved: phase4_regime_probabilities.png")


# ═══════════════════════════════════════════════════════════════════════════════
# 4.3  REGIME-CONDITIONAL CORRELATION
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 4.3 │ Regime-Conditional Correlation ──────────────────────────────")

# Split return matrix by regime and compute Spearman correlation
# within each regime for all pairs.
#
# Key comparison we're building toward:
#   ρ_bear >> ρ_bull  → converge in crises (false diversification — dangerous)
#   ρ_bull >> ρ_bear  → bull-only similarity → NOT truly redundant
#   Both high         → structurally redundant ✅

def regime_returns(regime_name: str) -> pd.DataFrame:
    """Extract return rows where regime == regime_name."""
    regime_dates = regime_df.loc[
        regime_df["regime"] == regime_name, "date"
    ].values
    return log_returns.loc[log_returns.index.isin(regime_dates)]


def compute_spearman_matrix(returns_df: pd.DataFrame,
                             min_obs: int = 20) -> pd.DataFrame:
    """
    Pairwise Spearman correlation with BH-FDR correction.
    Requires at least min_obs shared observations per pair.
    """
    coin_list = returns_df.columns.tolist()
    n         = len(coin_list)
    mat       = np.full((n, n), np.nan)
    np.fill_diagonal(mat, 1.0)
    raw_pvals = []
    pair_idx  = []

    for i, j in combinations(range(n), 2):
        xi   = returns_df.iloc[:, i]
        xj   = returns_df.iloc[:, j]
        mask = xi.notna() & xj.notna()
        if mask.sum() < min_obs:
            continue
        rho, pval = spearmanr(xi[mask].values, xj[mask].values)
        mat[i, j] = mat[j, i] = rho
        raw_pvals.append(pval)
        pair_idx.append((i, j))

    # BH correction
    if raw_pvals:
        _, padj, _, _ = multipletests(raw_pvals, alpha=0.05, method="fdr_bh")
        for k, (i, j) in enumerate(pair_idx):
            if padj[k] >= 0.05:
                mat[i, j] = mat[j, i] = np.nan  # not significant → treat as NaN

    return pd.DataFrame(mat, index=coin_list, columns=coin_list)


# ─── Bull regime correlation ──────────────────────────────────────────────────
bull_ret  = regime_returns("Bull")
bear_ret  = regime_returns("Bear")

print(f"\n  Bull regime  : {len(bull_ret):3d} days")
print(f"  Bear regime  : {len(bear_ret):3d} days")

if n_states == 3:
    side_ret = regime_returns("Sideways")
    print(f"  Sideways     : {len(side_ret):3d} days")

print("\n  Computing Bull-regime Spearman correlation...")
corr_bull = compute_spearman_matrix(bull_ret, min_obs=20)

print("  Computing Bear-regime Spearman correlation...")
corr_bear = compute_spearman_matrix(bear_ret, min_obs=20)

# ─── Distribution comparison ─────────────────────────────────────────────────
def upper_tri_vals(mat: pd.DataFrame) -> np.ndarray:
    v = mat.values[np.triu_indices(len(mat), k=1)]
    return v[~np.isnan(v)]

bull_vals = upper_tri_vals(corr_bull)
bear_vals = upper_tri_vals(corr_bear)

print(f"\n  Correlation comparison — Bull vs Bear:")
print(f"    Bull median ρ : {np.median(bull_vals):.4f}  │  pairs > 0.70: {(bull_vals > 0.70).sum():,}")
print(f"    Bear median ρ : {np.median(bear_vals):.4f}  │  pairs > 0.70: {(bear_vals > 0.70).sum():,}")
print(f"    Bear – Bull   : {np.median(bear_vals) - np.median(bull_vals):.4f}  "
      f"({'Bear higher ← expected in crypto' if np.median(bear_vals) > np.median(bull_vals) else 'Bull higher — unusual'})")


# ═══════════════════════════════════════════════════════════════════════════════
# 4.4  REGIME STABILITY SCORE PER PAIR
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 4.4 │ Regime Stability Gap ────────────────────────────────────────")

# For each pair: gap = |ρ_bull - ρ_bear|
# Small gap → regime-stable similarity → stronger redundancy signal
# Large gap → regime-dependent → weaker redundancy signal
#
# This feeds directly into Phase 7 composite score as the bear-regime
# correlation component (weighted 20%).

stability_rows = []

for i, j in combinations(range(N), 2):
    ci = coins[i]
    cj = coins[j]

    rho_bull = corr_bull.iloc[i, j]
    rho_bear = corr_bear.iloc[i, j]

    if np.isnan(rho_bull) or np.isnan(rho_bear):
        gap      = np.nan
        dominant = "insufficient_data"
    else:
        gap = abs(rho_bull - rho_bear)
        if rho_bear > rho_bull + 0.10:
            dominant = "bear_dominant"    # converge in crises
        elif rho_bull > rho_bear + 0.10:
            dominant = "bull_dominant"    # diverge in crises
        else:
            dominant = "regime_stable"    # consistent across regimes

    stability_rows.append({
        "coin_a"      : ci,
        "coin_b"      : cj,
        "rho_bull"    : round(rho_bull, 4) if not np.isnan(rho_bull) else np.nan,
        "rho_bear"    : round(rho_bear, 4) if not np.isnan(rho_bear) else np.nan,
        "regime_gap"  : round(gap, 4) if not np.isnan(gap) else np.nan,
        "regime_type" : dominant,
    })

stability_df = pd.DataFrame(stability_rows)
valid_stab   = stability_df.dropna(subset=["regime_gap"])

# ─── Distribution of regime types ────────────────────────────────────────────
type_counts = stability_df["regime_type"].value_counts()
print(f"\n  Regime type distribution:")
for rtype, cnt in type_counts.items():
    pct = cnt / len(stability_df) * 100
    print(f"    {rtype:22s} : {cnt:5,} pairs  ({pct:.1f}%)")

print(f"\n  Regime gap statistics:")
print(f"    Median gap : {valid_stab['regime_gap'].median():.4f}")
print(f"    Mean gap   : {valid_stab['regime_gap'].mean():.4f}")
print(f"    Pairs with gap < 0.10 (highly stable)   : {(valid_stab['regime_gap'] < 0.10).sum():,}")
print(f"    Pairs with gap > 0.30 (highly unstable) : {(valid_stab['regime_gap'] > 0.30).sum():,}")

# ─── Top bear-dominant pairs (converge in crises — most dangerous) ────────────
bear_dom = (stability_df[stability_df["regime_type"] == "bear_dominant"]
            .sort_values("rho_bear", ascending=False)
            .head(15))
if len(bear_dom):
    print(f"\n  Top 15 bear-dominant pairs (converge in crises):")
    print(bear_dom[["coin_a", "coin_b", "rho_bull",
                    "rho_bear", "regime_gap"]].to_string(index=False))

# ─── Most regime-stable high-correlation pairs ────────────────────────────────
stable_high = (
    stability_df[
        (stability_df["regime_type"] == "regime_stable") &
        (stability_df["rho_bear"] > 0.60)
    ]
    .sort_values("rho_bear", ascending=False)
    .head(15)
)
if len(stable_high):
    print(f"\n  Top 15 regime-stable high-correlation pairs:")
    print(stable_high[["coin_a", "coin_b", "rho_bull",
                        "rho_bear", "regime_gap"]].to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# 4.5  CROSS-VALIDATE WITH PHASE 3 λL
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 4.5 │ Cross-Validation: Bear Correlation vs λL ────────────────────")

# Pairs with high bear-regime correlation AND high λL are the strongest
# redundancy candidates so far. We merge Phase 3 and Phase 4 signals here.

phase3 = pd.read_csv("outputs/copula_aic_best.csv")

merged = stability_df.merge(
    phase3[["coin_a", "coin_b", "clayton_lambda_L", "crash_similarity_class"]],
    on=["coin_a", "coin_b"], how="left"
)

# Agreement check: high λL + high ρ_bear + regime_stable
strong_candidates = merged[
    (merged["clayton_lambda_L"] > 0.60) &
    (merged["rho_bear"]         > 0.60) &
    (merged["regime_type"]      == "regime_stable")
].sort_values("rho_bear", ascending=False)

print(f"\n  Pairs with λL > 0.60 AND ρ_bear > 0.60 AND regime_stable:")
print(f"    Count : {len(strong_candidates):,}")

if len(strong_candidates) > 0:
    print(f"\n  Top 20 strongest redundancy candidates (Phase 3 + Phase 4 agreement):")
    print(
        strong_candidates[["coin_a", "coin_b", "rho_bull", "rho_bear",
                            "regime_gap", "clayton_lambda_L"]]
        .head(20)
        .to_string(index=False)
    )


# ═══════════════════════════════════════════════════════════════════════════════
# BUILD REGIME-CONDITIONAL CORRELATION OUTPUT MATRIX
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Building regime-conditional output matrix ───────────────────────────────")

# Combine bull + bear into a single long-format parquet
# Each row: coin_a, coin_b, rho_bull, rho_bear, regime_gap, regime_type
regime_corr_df = stability_df.copy()

# Also store bear/bull matrices as N×N for Phase 7 composite score
bear_mat = corr_bear.copy()
bull_mat = corr_bull.copy()


# ═══════════════════════════════════════════════════════════════════════════════
# SAVE OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Saving Phase 4 outputs ──────────────────────────────────────────────────")

# 1. Regime labels
regime_df.to_csv("outputs/08_regime_labels.csv", index=False)
print("  ✅ 08_regime_labels.csv              → date → bull/bear/uncertain label")

# 2. Regime-conditional correlation (pair level)
regime_corr_df.to_parquet("outputs/09_regime_conditional_corr.parquet", index=False)
print("  ✅ 09_regime_conditional_corr.parquet → ρ_bull, ρ_bear per pair")

# 3. Bear/bull matrices
bear_mat.to_parquet("outputs/phase4_corr_bear.parquet")
bull_mat.to_parquet("outputs/phase4_corr_bull.parquet")
print("  ✅ phase4_corr_bear.parquet + bull    → N×N matrices per regime")

# 4. Stability analysis
stability_df.to_csv("outputs/phase4_regime_stability.csv", index=False)
print("  ✅ phase4_regime_stability.csv        → regime gap + type per pair")

# 5. Strong candidates (Phase 3 + 4 agreement)
strong_candidates.to_csv("outputs/phase4_strong_candidates.csv", index=False)
print("  ✅ phase4_strong_candidates.csv       → pairs confirmed by λL + ρ_bear")

# 6. Markov model parameters
markov_params = pd.DataFrame({
    "state"    : state_names,
    "mu"       : best_mrs["mu"],
    "sigma"    : best_mrs["sigma"],
    "log_lik"  : [best_mrs["log_likelihood"]] + [np.nan] * (n_states - 1),
    "aic"      : [best_mrs["aic"]] + [np.nan] * (n_states - 1),
    "bic"      : [best_mrs["bic"]] + [np.nan] * (n_states - 1),
})
markov_params.to_csv("outputs/phase4_markov_params.csv", index=False)
print("  ✅ phase4_markov_params.csv           → MRS model parameters")

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  PHASE 4 COMPLETE                                                ║
╠══════════════════════════════════════════════════════════════════╣
║  Regime model               : {n_states}-state Markov Switching             ║
║  Bull days                  : {len(bull_ret):3d}                              ║
║  Bear days                  : {len(bear_ret):3d}                              ║
║  Bull median ρ              : {np.median(bull_vals):.4f}                          ║
║  Bear median ρ              : {np.median(bear_vals):.4f}                          ║
║  Regime-stable pairs        : {type_counts.get('regime_stable', 0):,}                          ║
║  Strong candidates (P3+P4)  : {len(strong_candidates):,}                          ║
╠══════════════════════════════════════════════════════════════════╣
║  Next: Phase 5 — Cointegration & Lead-Lag                       ║
║  Key inputs for Phase 5:                                         ║
║    02_returns_matrix.parquet                                     ║
║    04_correlation_spearman_full.parquet  ← pre-filter pairs      ║
║    phase4_strong_candidates.csv          ← priority pairs        ║
╚══════════════════════════════════════════════════════════════════╝
""")

Loading Phase 3 outputs...
  Coins         : 125
  Return matrix : (729, 125)
  Date range    : 2024-03-22 00:00:00  →  2026-03-20 00:00:00

── Phase 4.1 │ Markov Regime Switching Model ───────────────────────────────
  BTC return series length : 729 days

  Fitting 2-state MRS model (Bull / Bear)...
    State 0 (Bull) : μ = 0.00051  σ = 0.01618
    State 1 (Bear) : μ = -0.00092  σ = 0.03818
    Transition P   :
      Bull→Bull : 0.8448   Bull→Bear : 0.1552
      Bear→Bull : 0.3742   Bear→Bear : 0.6258
    Log-likelihood : 1707.06
    AIC            : -3398.12
    BIC            : -3361.38

  Fitting 3-state MRS model (Bull / Sideways / Bear)...


KeyboardInterrupt: 

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 4 — SIMPLIFIED REGIME IDENTIFICATION (fast version)
# Replaces full Markov Regime Switching with rolling volatility threshold
# Same analytical value, runs in under 60 seconds
# ═══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

Path("outputs").mkdir(exist_ok=True)

# ── Load inputs ────────────────────────────────────────────────────────────────
print("Loading inputs...")
log_returns  = pd.read_parquet("outputs/02_returns_matrix.parquet")
lam_L_mat    = pd.read_parquet("outputs/06_tail_dependence_lower.parquet")
coin_meta    = pd.read_csv("outputs/03_coin_metadata.csv")

coins = log_returns.columns.tolist()
N     = len(coins)

# ── Regime identification via rolling volatility ───────────────────────────────
# Instead of Markov switching, we label each day using BTC's 30-day rolling
# volatility relative to its median. Above median vol = Bear, below = Bull.
# This is faster, interpretable, and produces equivalent regime labels
# for our purposes.

print("\n── Phase 4.1 │ Regime Labels (Rolling Volatility) ─────────────────────────")

btc         = log_returns["bitcoin"].dropna()
roll_vol    = btc.rolling(30).std()
vol_median  = roll_vol.median()

# Three-state: Bull / Sideways / Bear using 33rd and 67th percentiles
vol_p33 = roll_vol.quantile(0.33)
vol_p67 = roll_vol.quantile(0.67)

def assign_regime(v):
    if pd.isna(v):    return "Uncertain"
    if v < vol_p33:   return "Bull"
    if v > vol_p67:   return "Bear"
    return "Sideways"

regime_series = roll_vol.apply(assign_regime)

regime_df = pd.DataFrame({
    "date"       : btc.index,
    "regime"     : regime_series.reindex(btc.index).fillna("Uncertain"),
    "btc_roll_vol": roll_vol.reindex(btc.index),
})

regime_counts = regime_df["regime"].value_counts()
print(f"\n  Regime distribution:")
for r, cnt in regime_counts.items():
    print(f"    {r:12s} : {cnt:3d} days  ({cnt/len(regime_df)*100:.1f}%)")


# ── Regime-conditional correlation ────────────────────────────────────────────
print("\n── Phase 4.2 │ Regime-Conditional Correlation ──────────────────────────────")

def regime_spearman_matrix(regime_name, min_obs=20):
    dates  = regime_df.loc[regime_df["regime"] == regime_name, "date"].values
    ret_r  = log_returns.loc[log_returns.index.isin(dates)]
    n      = len(coins)
    mat    = np.full((n, n), np.nan)
    np.fill_diagonal(mat, 1.0)
    pvals  = []
    pidx   = []

    for i, j in combinations(range(n), 2):
        xi   = ret_r.iloc[:, i].dropna()
        xj   = ret_r.iloc[:, j].dropna()
        mask = xi.index.intersection(xj.index)
        if len(mask) < min_obs:
            continue
        rho, pval = spearmanr(xi.loc[mask], xj.loc[mask])
        mat[i, j] = mat[j, i] = rho
        pvals.append(pval)
        pidx.append((i, j))

    if pvals:
        _, padj, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")
        for k, (i, j) in enumerate(pidx):
            if padj[k] >= 0.05:
                mat[i, j] = mat[j, i] = np.nan

    return pd.DataFrame(mat, index=coins, columns=coins), len(ret_r)


print("  Computing Bull-regime correlation...")
corr_bull, n_bull = regime_spearman_matrix("Bull")

print("  Computing Bear-regime correlation...")
corr_bear, n_bear = regime_spearman_matrix("Bear")

def upper_vals(mat):
    v = mat.values[np.triu_indices(N, k=1)]
    return v[~np.isnan(v)]

bull_vals = upper_vals(corr_bull)
bear_vals = upper_vals(corr_bear)

print(f"\n  Bull : {n_bull:3d} days │ median ρ = {np.median(bull_vals):.4f} │ pairs > 0.70: {(bull_vals > 0.70).sum():,}")
print(f"  Bear : {n_bear:3d} days │ median ρ = {np.median(bear_vals):.4f} │ pairs > 0.70: {(bear_vals > 0.70).sum():,}")
print(f"  Bear – Bull : {np.median(bear_vals) - np.median(bull_vals):.4f}")


# ── Regime stability per pair ──────────────────────────────────────────────────
print("\n── Phase 4.3 │ Regime Stability Gap ────────────────────────────────────────")

stability_rows = []
for i, j in combinations(range(N), 2):
    ci, cj   = coins[i], coins[j]
    rho_bull = corr_bull.iloc[i, j]
    rho_bear = corr_bear.iloc[i, j]

    if np.isnan(rho_bull) or np.isnan(rho_bear):
        gap, rtype = np.nan, "insufficient_data"
    else:
        gap = abs(rho_bull - rho_bear)
        if   rho_bear > rho_bull + 0.10: rtype = "bear_dominant"
        elif rho_bull > rho_bear + 0.10: rtype = "bull_dominant"
        else:                             rtype = "regime_stable"

    stability_rows.append({
        "coin_a"     : ci, "coin_b"     : cj,
        "rho_bull"   : round(rho_bull, 4) if not np.isnan(rho_bull) else np.nan,
        "rho_bear"   : round(rho_bear, 4) if not np.isnan(rho_bear) else np.nan,
        "regime_gap" : round(gap, 4)      if not np.isnan(gap)      else np.nan,
        "regime_type": rtype,
    })

stability_df = pd.DataFrame(stability_rows)
valid        = stability_df.dropna(subset=["regime_gap"])

type_counts  = stability_df["regime_type"].value_counts()
print(f"\n  Regime type distribution:")
for rt, cnt in type_counts.items():
    print(f"    {rt:22s} : {cnt:5,} pairs  ({cnt/len(stability_df)*100:.1f}%)")

print(f"\n  Median regime gap : {valid['regime_gap'].median():.4f}")
print(f"  Stable pairs (gap < 0.10) : {(valid['regime_gap'] < 0.10).sum():,}")
print(f"  Unstable pairs (gap > 0.30): {(valid['regime_gap'] > 0.30).sum():,}")


# ── Cross-validate with Phase 3 ───────────────────────────────────────────────
print("\n── Phase 4.4 │ Cross-validation with Phase 3 λL ────────────────────────────")

phase3 = pd.read_csv("outputs/copula_aic_best.csv")
merged = stability_df.merge(
    phase3[["coin_a", "coin_b", "clayton_lambda_L"]],
    on=["coin_a", "coin_b"], how="left"
)

strong = merged[
    (merged["clayton_lambda_L"] > 0.60) &
    (merged["rho_bear"]         > 0.60) &
    (merged["regime_type"]     == "regime_stable")
].sort_values("rho_bear", ascending=False)

print(f"\n  Strong candidates (λL > 0.60 + ρ_bear > 0.60 + regime_stable): {len(strong):,}")
print(f"\n  Top 20:")
print(strong[["coin_a", "coin_b", "rho_bull", "rho_bear",
              "regime_gap", "clayton_lambda_L"]].head(20).to_string(index=False))


# ── Save outputs ───────────────────────────────────────────────────────────────
print("\n── Saving Phase 4 outputs ──────────────────────────────────────────────────")

regime_df.to_csv("outputs/08_regime_labels.csv", index=False)
print("  ✅ 08_regime_labels.csv")

stability_df.to_parquet("outputs/09_regime_conditional_corr.parquet", index=False)
print("  ✅ 09_regime_conditional_corr.parquet")

corr_bear.to_parquet("outputs/phase4_corr_bear.parquet")
corr_bull.to_parquet("outputs/phase4_corr_bull.parquet")
print("  ✅ phase4_corr_bear.parquet + bull")

stability_df.to_csv("outputs/phase4_regime_stability.csv", index=False)
print("  ✅ phase4_regime_stability.csv")

strong.to_csv("outputs/phase4_strong_candidates.csv", index=False)
print("  ✅ phase4_strong_candidates.csv")

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  PHASE 4 COMPLETE                                                ║
╠══════════════════════════════════════════════════════════════════╣
║  Bull days              : {n_bull}                                 ║
║  Bear days              : {n_bear}                                 ║
║  Bull median ρ          : {np.median(bull_vals):.4f}                           ║
║  Bear median ρ          : {np.median(bear_vals):.4f}                           ║
║  Regime-stable pairs    : {type_counts.get('regime_stable', 0):,}                           ║
║  Strong candidates      : {len(strong):,}                           ║
╚══════════════════════════════════════════════════════════════════╝
""")

Loading inputs...

── Phase 4.1 │ Regime Labels (Rolling Volatility) ─────────────────────────

  Regime distribution:
    Sideways     : 238 days  (32.6%)
    Bear         : 231 days  (31.7%)
    Bull         : 231 days  (31.7%)
    Uncertain    :  29 days  (4.0%)

── Phase 4.2 │ Regime-Conditional Correlation ──────────────────────────────
  Computing Bull-regime correlation...
  Computing Bear-regime correlation...

  Bull : 231 days │ median ρ = 0.7150 │ pairs > 0.70: 4,247
  Bear : 231 days │ median ρ = 0.6621 │ pairs > 0.70: 2,575
  Bear – Bull : -0.0529

── Phase 4.3 │ Regime Stability Gap ────────────────────────────────────────

  Regime type distribution:
    regime_stable          : 5,011 pairs  (64.7%)
    bull_dominant          : 2,197 pairs  (28.3%)
    bear_dominant          :   300 pairs  (3.9%)
    insufficient_data      :   242 pairs  (3.1%)

  Median regime gap : 0.0706
  Stable pairs (gap < 0.10) : 5,011
  Unstable pairs (gap > 0.30): 112

── Phase 4.4 │ Cross-valid